In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Modelling
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.model_selection import train_test_split,RandomizedSearchCV

In [19]:
df=pd.read_csv('data/stud.csv')

In [20]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [21]:
df['average']=(df['math_score']+df['reading_score']+df['writing_score'])/3

In [22]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,average
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667
1,female,group C,some college,standard,completed,69,90,88,82.333333
2,female,group B,master's degree,standard,none,90,95,93,92.666667
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333
4,male,group C,some college,standard,none,76,78,75,76.333333


In [23]:
X=df.iloc[:,:-1]
Y=df['average']

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
nominal=[i for i in X.columns if X[i].dtype=='str']
numerical=[i for i in X.columns if i not in nominal]
preprocessor=ColumnTransformer(transformers=[
    ('ohe',OneHotEncoder(),nominal),
    ('ss',StandardScaler(),numerical)
])

In [25]:
X=preprocessor.fit_transform(X)
X.shape

(1000, 20)

In [26]:
X_Train,X_Test,Y_Train,Y_Test=train_test_split(X,Y,test_size=0.2,random_state=42)
X_Train.shape

(800, 20)

### Create an evaluate function to give all metrics after training model

In [27]:
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=root_mean_squared_error(true,predicted)
    r_squared=r2_score(true,predicted)
    return mae,mse,rmse,r_squared

In [30]:
models={
    'Linear Regressor':LinearRegression(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'KNR':KNeighborsRegressor(),
    'Decision Tree':DecisionTreeRegressor(),
    'RandomForest':RandomForestRegressor(),
    'SVR':SVR(),
    'AdaBoost':AdaBoostRegressor(),
    'XGBoost':XGBRegressor(),
    'CatBoost':CatBoostRegressor(verbose=False)
}
model_list=[]
r2_list=[]
for m in range(len(models)):
    model=list(models.values())[m]
    model.fit(X_Train,Y_Train)

    y_train_pred=model.predict(X_Train)
    y_test_pred=model.predict(X_Test)

    model_train_mae,model_train_mse,model_train_rmse,model_train_r2=evaluate_model(Y_Train,y_train_pred)
    model_test_mae,model_test_mse,model_test_rmse,model_test_r2=evaluate_model(Y_Test,y_test_pred)

    print(list(models.keys())[m])
    model_list.append(list(models.keys())[m])

    print('Model performance for training set')
    print(f'Root Mean Squared Error : {model_train_rmse:.4f}')
    print(f'Mean Absolute Error : {model_train_mae:.4f}')
    print(f'Mean Squared Error : {model_train_mse:.4f}')
    print(f'R2_Score : {model_train_r2:.4f}')

    print('------------------------------------')
    
    print('Model performance for testing set')
    print(f'Root Mean Squared Error : {model_test_rmse:.4f}')
    print(f'Mean Absolute Error : {model_test_mae:.4f}')
    print(f'Mean Squared Error : {model_test_mse:.4f}')
    print(f'R2_Score : {model_test_r2:.4f}')
    r2_list.append(model_test_r2)
    print('\n')

Linear Regressor
Model performance for training set
Root Mean Squared Error : 0.0000
Mean Absolute Error : 0.0000
Mean Squared Error : 0.0000
R2_Score : 1.0000
------------------------------------
Model performance for testing set
Root Mean Squared Error : 0.0000
Mean Absolute Error : 0.0000
Mean Squared Error : 0.0000
R2_Score : 1.0000


Ridge
Model performance for training set
Root Mean Squared Error : 0.0080
Mean Absolute Error : 0.0065
Mean Squared Error : 0.0001
R2_Score : 1.0000
------------------------------------
Model performance for testing set
Root Mean Squared Error : 0.0088
Mean Absolute Error : 0.0067
Mean Squared Error : 0.0001
R2_Score : 1.0000


Lasso
Model performance for training set
Root Mean Squared Error : 1.0645
Mean Absolute Error : 0.8506
Mean Squared Error : 1.1332
R2_Score : 0.9943
------------------------------------
Model performance for testing set
Root Mean Squared Error : 1.1142
Mean Absolute Error : 0.8769
Mean Squared Error : 1.2414
R2_Score : 0.9942



In [31]:
pd.DataFrame({"models":model_list,"r2_score":r2_list})

,models,r2_score
0,Linear Regressor,1.000000
1,Ridge,1.000000
2,Lasso,0.994209
3,KNR,0.956016
4,Decision Tree,0.988877
5,RandomForest,0.993669
6,SVR,0.863902
7,AdaBoost,0.977929
8,XGBoost,0.994748
9,CatBoost,0.993447
